In [0]:
import subprocess
subprocess.check_call(['pip', 'install', 'faker', 'holidays'])

#from utils import save_to_parquet
import numpy as np, pandas as pd, random
from faker import Faker
import holidays

np.random.seed(42)
random.seed(42)
fake = Faker()
Faker.seed(42)

# Get catalog and schema from notebook widgets / job parameters
# Widgets are automatically created from base_parameters in the workflow
# For local or ad‑hoc runs, defaults are used
dbutils.widgets.text("catalog", "hls_glucosphere", "Catalog")
dbutils.widgets.text("schema", "cgm", "Schema")

try:
    CATALOG = dbutils.widgets.get("catalog")
    SCHEMA = dbutils.widgets.get("schema")
except Exception as e:
    raise RuntimeError(f"Widget lookup failed, catalog/schema not passed to notebook: {e}")

print(f"Using catalog: {CATALOG}, schema: {SCHEMA}")

# Ensure Unity Catalog structure exists for data storage
print("Setting up Unity Catalog structure...")
try:
    # Ensure catalog exists (if you don't have permission to create catalogs,
    # comment this out and create the catalog once via SQL instead)
    #spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
    print(f"✓ Catalog '{CATALOG}' ready")
    
    # Ensure schema exists
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
    print(f"✓ Schema '{SCHEMA}' ready")
    
    # NOW set current catalog and schema (after creating them)
    spark.sql(f"USE CATALOG {CATALOG}")
    spark.sql(f"USE SCHEMA {SCHEMA}")
    
    # Ensure volume exists
    spark.sql(f"""
        CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.landing_zone
        COMMENT 'Landing zone for MedTech raw data files'
    """)
    print("✓ Volume 'landing_zone' ready")
    
    # Create subdirectories for all datasets using dbutils
    volume_base = f"/Volumes/{CATALOG}/{SCHEMA}/landing_zone"
    datasets = [
        "raw_patient_registry",
        "raw_device_telemetry_stream"
    ]
    
    for dataset in datasets:
        dataset_path = f"{volume_base}/{dataset}"
        try:
            dbutils.fs.mkdirs(dataset_path)
            print(f"✓ Volume directory '{dataset}' created")
        except Exception as dir_error:
            print(f"⚠ Could not create directory {dataset}: {dir_error}")
    
except Exception as e:
    print(f"⚠ Could not create Unity Catalog structure: {e}")
    print("  You may need to create catalog/schema/volume manually before running this script")


In [0]:
from pyspark.sql.functions import (
    when, col, lit, lag, min as spark_min, rand
)
from pyspark.sql.window import Window

device_models_list = ["Alpha", "Beta", "Gamma", "Delta", "Epsilon", "Zeta"]

def make_device_firmware():
    # Base incident table
    df = spark.table("hls_glucosphere.cgm.pseudo_incident_7d_labeled_v20260105")

    # Patient-device mapping
    patient_device_df = (
        spark.table("hls_glucosphere.cgm.patient_device")
        .select("patient_id", "device_id")
        .dropDuplicates(["patient_id", "device_id"])
    )

    # Join to get patient_device.device_id by patient_id
    df = df.join(patient_device_df, on="patient_id", how="left")

    # Assign one random device_model per device_id with given probabilities
    # Use a helper DataFrame of unique device_ids
    device_ids_df = df.select("device_id").dropna().dropDuplicates()

    # Add a uniform random number per device_id
    device_ids_df = device_ids_df.withColumn("u", rand(seed=42))

    # Map uniform [0,1) to categorical with your probabilities:
    # p = [0.22, 0.22, 0.19, 0.16, 0.13, 0.08]
    device_ids_df = device_ids_df.withColumn(
        "device_model",
        when(col("u") < 0.22, lit("Alpha"))
        .when(col("u") < 0.44, lit("Beta"))
        .when(col("u") < 0.63, lit("Gamma"))
        .when(col("u") < 0.79, lit("Delta"))
        .when(col("u") < 0.92, lit("Epsilon"))
        .otherwise(lit("Zeta"))
    ).drop("u")

    # Join device_model back to main df
    df = df.join(device_ids_df, on="device_id", how="left")

    firmware_version = (
        when(col("has_incident") == 0, lit(3.14))
        .when(
            (col("has_incident") == 1)
            & (col("time") < lit("2026-01-07T14:10:00.000+00:00")),
            lit(3.14),
        )
        .when(
            (col("has_incident") == 1)
            & (col("time") < lit("2026-01-07T17:00:00.000+00:00")),
            lit(4),
        )
        .when(
            (col("has_incident") == 1)
            & (col("time") < lit("2026-01-09T17:00:00.000+00:00")),
            lit(3.14),
        )
        .when(
            (col("has_incident") == 1)
            & (col("time") > lit("2026-01-09T16:55:00.000+00:00")),
            lit(4.10),
        )
        .otherwise(None)
    )

    base_df = df.select(
        col("patient_id"),
        col("device_id"),
        col("device_model"),
        col("time"),
        col("has_incident"),
        firmware_version.alias("firmware_version"),
    )

    # For has_incident = 0, keep only the earliest time per device_id
    incident0_df = base_df.filter(col("has_incident") == 0)
    incident0_df = incident0_df.withColumn(
        "min_time", spark_min("time").over(Window.partitionBy("device_id"))
    ).filter(col("time") == col("min_time")).drop("min_time")

    # For has_incident = 1, keep only rows where firmware_version changes for a device_id
    incident1_df = base_df.filter(col("has_incident") == 1)
    window = Window.partitionBy("device_id").orderBy("time")
    incident1_df = incident1_df.withColumn(
        "prev_firmware_version", lag("firmware_version").over(window)
    ).filter(
        (col("firmware_version") != col("prev_firmware_version"))
        | col("prev_firmware_version").isNull()
    ).drop("prev_firmware_version")

    result_df = incident0_df.unionByName(incident1_df).select(
        "patient_id", "device_id", "device_model", "time", "firmware_version"
    )

    return result_df


In [0]:
from pyspark.sql.functions import (
    when, col, lit, lag, lead, min as spark_min, rand, coalesce
)
from pyspark.sql.window import Window

device_models_list = ["Alpha", "Beta", "Gamma", "Delta", "Epsilon", "Zeta"]

def make_device_firmware():
    # Base incident table
    df = spark.table("hls_glucosphere.cgm.pseudo_incident_7d_labeled_v20260105")

    # Patient-device mapping
    patient_device_df = (
        spark.table("hls_glucosphere.cgm.patient_device")
        .select("patient_id", "device_id")
        .dropDuplicates(["patient_id", "device_id"])
    )

    # Join to get patient_device.device_id by patient_id
    df = df.join(patient_device_df, on="patient_id", how="left")

    # Assign one random device_model per device_id with given probabilities
    device_ids_df = df.select("device_id").dropna().dropDuplicates()

    # Add a uniform random number per device_id
    device_ids_df = device_ids_df.withColumn("u", rand(seed=42))

    # Map uniform [0,1) to categorical with your probabilities:
    # p = [0.22, 0.22, 0.19, 0.16, 0.13, 0.08]
    device_ids_df = device_ids_df.withColumn(
        "device_model",
        when(col("u") < 0.22, lit("Alpha"))
        .when(col("u") < 0.44, lit("Beta"))
        .when(col("u") < 0.63, lit("Gamma"))
        .when(col("u") < 0.79, lit("Delta"))
        .when(col("u") < 0.92, lit("Epsilon"))
        .otherwise(lit("Zeta"))
    ).drop("u")

    # Join device_model back to main df
    df = df.join(device_ids_df, on="device_id", how="left")

    firmware_version = (
        when(col("has_incident") == 0, lit(3.14))
        .when(
            (col("has_incident") == 1)
            & (col("time") < lit("2026-01-07T14:10:00.000+00:00")),
            lit(3.14),
        )
        .when(
            (col("has_incident") == 1)
            & (col("time") < lit("2026-01-07T17:00:00.000+00:00")),
            lit(4),
        )
        .when(
            (col("has_incident") == 1)
            & (col("time") < lit("2026-01-09T17:00:00.000+00:00")),
            lit(3.14),
        )
        .when(
            (col("has_incident") == 1)
            & (col("time") > lit("2026-01-09T16:55:00.000+00:00")),
            lit(4.10),
        )
        .otherwise(None)
    )

    base_df = df.select(
        col("patient_id"),
        col("device_id"),
        col("device_model"),
        col("time").alias("start_time"),
        col("has_incident"),
        firmware_version.alias("firmware_version"),
    )

    # For has_incident = 0, keep only the earliest start_time per device_id
    incident0_df = base_df.filter(col("has_incident") == 0)
    incident0_df = incident0_df.withColumn(
        "min_time", spark_min("start_time").over(Window.partitionBy("device_id"))
    ).filter(col("start_time") == col("min_time")).drop("min_time")

    # For has_incident = 1, keep only rows where firmware_version changes for a device_id
    incident1_df = base_df.filter(col("has_incident") == 1)
    window = Window.partitionBy("device_id").orderBy("start_time")
    incident1_df = incident1_df.withColumn(
        "prev_firmware_version", lag("firmware_version").over(window)
    ).filter(
        (col("firmware_version") != col("prev_firmware_version"))
        | col("prev_firmware_version").isNull()
    ).drop("prev_firmware_version")

    # Union the two incident sets
    result_df = incident0_df.unionByName(incident1_df)

    # Compute end_time as next start_time per device_id
    window_time = Window.partitionBy("device_id").orderBy("start_time")
    result_df = result_df.withColumn(
        "end_time_raw", lead("start_time").over(window_time)
    )

    # Replace null with a far-future timestamp so between-joins work
    far_future = lit("9999-12-31T23:59:59.000+00:00")
    result_df = result_df.withColumn(
        "end_time", coalesce(col("end_time_raw"), far_future)
    ).drop("end_time_raw")

    # Select final columns
    result_df = result_df.select(
        "patient_id",
        "device_id",
        "device_model",
        "start_time",
        "end_time",
        "firmware_version",
    )

    return result_df

In [0]:
from pyspark.sql.functions import count_distinct, collect_list

raw_data_name = "raw_device_telemetry_stream"
path = f"{volume_base}/{raw_data_name}"  

raw_device_telemetry_stream = make_device_firmware()
raw_device_telemetry_stream.write.mode("overwrite").parquet(path)
display(raw_device_telemetry_stream)

In [0]:
df = raw_device_telemetry_stream.filter(col("patient_id") == "PSEUDO_0000096")
display(df)

In [0]:
%sql
select time, has_incident, avg(glucose_true), avg(glucose_observed)
from hls_glucosphere.cgm.pseudo_incident_7d_labeled_v20260105
group by time, has_incident
order by time asc

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
select min(time), max(time)
from hls_glucosphere.cgm.pseudo_incident_7d_labeled_v20260105

In [0]:
%sql
select time, sum(case when has_incident = 1 then 1 else 0 end) as has_incident, sum(case when has_incident = 0 then 1 else 0 end) as no_incident
from hls_glucosphere.cgm.pseudo_incident_7d_labeled_v20260105
group by 1
order by time asc

In [0]:
%sql
select * from hls_glucosphere.cgm.pseudo_incident_7d_labeled_v20260105
where patient_id = 'PSEUDO_0000096'